# Отток клиентов

После исправления методики прежние метрики требуют повторного расчёта. Сохранённые outputs очищены; историческая версия доступна в Git. Данные не включены в репозиторий. См. METHODOLOGY.md.

"Бета-Банк" столкнулся с проблемой оттока клиентов. По мнению банковских маркетологов, сохранять текущих клиентов дешевле, чем привлекать новых. 

Нам поставлена задача спрогнозировать, уйдёт клиент из банка в ближайшее время или нет. Для проведения анализа были предоставлены исторические данные о поведении клиентов и расторжении договоров с банком.

**Цель** - построить модель с предельно большим значением F1-меры (не менее 0.59).

Для решения поставленной задачи необходимо выделить следующие этапы:

1. Подготовка данных.
2. Исследование баланса классов.
3. Изучение модели без учета дисбаланса.
4. Создание балансированной модели.
5. Проведение тестирования.
6. Исследование метрики AUC-ROC.

<h1>Содержание<span class="tocSkip"></span></h1>
<div<span><a href="#Подготовка-данных" data-toc-modified-id="Подготовка-данных-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка данных</a></span></li><li><span><a href="#Исследование-задачи" data-toc-modified-id="Исследование-задачи-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Исследование задачи</a></span></li><li><span><a href="#Борьба-с-дисбалансом" data-toc-modified-id="Борьба-с-дисбалансом-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Борьба с дисбалансом</a></span></li><li><span><a href="#Тестирование-модели" data-toc-modified-id="Тестирование-модели-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Тестирование модели</a></span></li><li><span><a href="#Чек-лист-готовности-проекта" data-toc-modified-id="Чек-лист-готовности-проекта-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Чек-лист готовности проекта</a></span></li></ul></div>

## Подготовка данных

In [ ]:
# загрузим все необходимые библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import auc
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler

from pathlib import Path
import os
DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))


In [ ]:
df = pd.read_csv(DATA_DIR / 'Churn.csv')
if df['CustomerId'].isna().any() or df['CustomerId'].duplicated().any():
    raise ValueError('Ожидалась одна строка на клиента')

In [ ]:
# посмотрим на 1-ые пять строчек таблицы
df.head()

In [ ]:
# удалим ненужные столбцы RowNumber, Customerld и Surname, которые не понадобятся в дальнейшем анализе
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

In [ ]:
# посмотрим на общую информацию о таблице
df.info()

Пропуски Tenure заполняются медианой обучающей выборки после разбиения.

In [ ]:
# Медиана Tenure рассчитывается после разделения, только на train.

In [ ]:
# проверим данные на наличие дубликатов
df.duplicated().sum()

Дубликатов не обнаружено.

In [ ]:
# Кодирование выполняется после разделения, с колонками из train.
df_ohe = df.copy()

**Вывод**

Таблица с анализируемыми данными содержит 14 признаков и 10000 объектов. Согласно приложенной документации:

- RowNumber — индекс строки в данных;
- CustomerId — уникальный идентификатор клиента;
- Surname — фамилия;
- CreditScore — кредитный рейтинг;
- Geography — страна проживания;
- Gender — пол;
- Age — возраст;
- Tenure — сколько лет человек является клиентом банка;
- Balance — баланс на счёте;
- NumOfProducts — количество продуктов банка, используемых клиентом;
- HasCrCard — наличие кредитной карты;
- IsActiveMember — активность клиента;
- EstimatedSalary — предполагаемая зарплата;
- Exited — факт ухода клиента.

В процессе предобработки данных пропуски в столбце Tenure были заменены на медианное значение. Также были удалены столбцы RowNumber, CustomerId и Surname, так они не понадобятся нам для дальнейшего анализа. В качестве целевого признака будет выступать Exited.

## Исследование задачи

In [ ]:
# выделим в нашей выборке признаки и целевой признак
features = df_ohe.drop(['Exited'], axis=1)
target = df_ohe['Exited']

In [ ]:
# разделим выборку на обучающую и валидационную
features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=0.4,
                                                                             random_state=12345, stratify=target)

In [ ]:
# выделим 50% валидационной выборки на тестовую
features_valid, features_test, target_valid, target_test = train_test_split(features_valid, target_valid, test_size=0.5,
                                                                              random_state=12345, stratify=target_valid)
# Train-only preprocessing: test/validation не участвуют в вычислении медианы.
tenure_median = features_train['Tenure'].median()
if pd.isna(tenure_median):
    raise ValueError('Tenure полностью отсутствует в обучении')
features_train = features_train.copy()
features_valid = features_valid.copy()
features_test = features_test.copy()
for frame in (features_train, features_valid, features_test):
    frame['Tenure'] = frame['Tenure'].fillna(tenure_median)

features_train = pd.get_dummies(features_train, drop_first=True)
# Reindex enforces the training vocabulary; unseen categories map to the baseline.
features_valid = pd.get_dummies(features_valid).reindex(columns=features_train.columns, fill_value=0)
features_test = pd.get_dummies(features_test).reindex(columns=features_train.columns, fill_value=0)


In [ ]:
features_test

In [ ]:
# проверим целевой признак на сбалансированность классов
class_frequency = df['Exited'].value_counts(normalize=True)
class_frequency

In [ ]:
# построим диаграмму частот классов в целевом признаке
class_frequency.plot(kind='bar', title='Распределение классов целевого признака', grid=True, figsize=(10, 6))
plt.xlabel('Классы Tuner')
plt.ylabel('Частота признака')
plt.show()

Видим, что соотношение классов далеко от идеального 1 к 1. В указанной выборке представены данные о клиентах, которые покинули банк, в количестве 20%. При этом осталось 80% клиентов.

Изучим модели без учета дисбаланса. Но перед этим проведем стандартизацию численных признаков.

In [ ]:
numeric = list(features_train.columns)

In [ ]:
# стандартизируем численные признаки
scaler = StandardScaler()
scaler.fit(features_train[numeric])
features_train[numeric] = scaler.transform(features_train[numeric])
features_valid[numeric] = scaler.transform(features_valid[numeric])

Построим модели без учета дисбаланса.

**Модель логистической регрессии**

In [ ]:
# построим модель логистической регрессии и оценим ее метрикой F1
model=LogisticRegression(random_state=12345, solver='liblinear')
model.fit(features_train, target_train)
predictions_regression = model.predict(features_valid)
print('F1-мера:', f1_score(target_valid, predictions_regression))

**Модель решающего дерева**

In [ ]:
# построим модель решающего дерева с перебором оптимальной глубины
best_tree_model=None
best_depth=0
best_f1=0

for depth in range(1, 20, 1):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    predictions_tree = model.predict(features_valid)
    print('Глубина дерева:', depth, '/ F1-мера:', f1_score(target_valid, predictions_tree))
    if f1_score(target_valid, predictions_tree) > best_f1:
        best_f1=f1_score(target_valid, predictions_tree)
        best_tree_model=model
        best_depth=depth
print('-------------------------------------------------------')        
print('F1 лучшей модели:', best_f1)
print('Лучшая глубина дерева:', best_depth)

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

**Модель случайного леса**

In [ ]:
# построим модель случайного леса с перебором оптимальной глубины
best_forest_model=None
best_depth=0
best_f1=0

for depth in range(1, 20, 1):
    model = RandomForestClassifier(random_state=12345, max_depth=depth, n_estimators=100)
    model.fit(features_train, target_train)
    predictions_forest = model.predict(features_valid)
    print('Глубина дерева:', depth, '/ F1-мера:', f1_score(target_valid, predictions_forest))
    if f1_score(target_valid, predictions_forest) > best_f1:
        best_forest_model=model
        best_f1=f1_score(target_valid, predictions_forest)
        best_depth=depth
print('-------------------------------------------------------')
print('F1 лучшей модели:', best_f1)
print('Лучшая глубина дерева:', best_depth)

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

In [ ]:
# построим модель случайного леса с перебором количества деревьев
best_forest_model=None
best_estimators=0
best_f1=0

for estimators in range(1, 100, 4):
    model = RandomForestClassifier(random_state=12345, max_depth=forest_depth, n_estimators=estimators)
    model.fit(features_train, target_train)
    predictions_forest = model.predict(features_valid)
    print('Количество деревьев:', estimators, '/ F1-мера:', f1_score(target_valid, predictions_forest))
    if f1_score(target_valid, predictions_forest) > best_f1:
        best_forest_model=model
        best_f1=f1_score(target_valid, predictions_forest)
        best_estimators=estimators
print('-------------------------------------------------------')
print('F1 лучшей модели:', best_f1)
print('Лучшее количество деревьев:', best_estimators)

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

**Вывод**

В процессе исследования данных было обнаружен дисбаланс классов целевого признака, что не позволяет эффективно обучить модель. Согласно изученной информации, 20% занимают данные о клиентах, которые покинули банк, и 80% - которые остались лояльны к банку.

Также были построены модели и посчитана F1-мера путем перебора различных гиперпараметров без учета дисбаланса. Самой эффективной моделью оказалась модель случайного леса.

## Борьба с дисбалансом

Для решения проблемы с дисбалансом применим три метода:
1. Взвешивание классов.
2. Увеличение выборки.
3. Уменьшение выборки.

**Взвешивание классов**

In [ ]:
# создадим функцию для расчета F1-меры после применения метода взвешивания классов
def calc_f1(model_ml, features, target):
    model=model_ml
    model.fit(features, target)
    predicted_valid = model.predict(features_valid)
    print('F1-мера:', f1_score(target_valid, predicted_valid))

In [ ]:
model_1 = LogisticRegression(random_state=12345, solver='liblinear', class_weight='balanced')
model_2 = DecisionTreeClassifier(**{**best_tree_model.get_params(), 'class_weight': 'balanced'})
model_3 = RandomForestClassifier(**{**best_forest_model.get_params(), 'class_weight': 'balanced'})

In [ ]:
# рассчитаем F1 для каждой модели
calc_f1(model_1, features_train, target_train)
calc_f1(model_2, features_train, target_train)
calc_f1(model_3, features_train, target_train)

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

**Увеличение выборки**

In [ ]:
# создадим функцию upsample для увеличения выборки
def upsample(features, target, repeat):
    features_zeros = features[target==0]
    features_ones = features[target==1]
    target_zeros = target[target==0]
    target_ones = target[target==1]
    
    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)
    
    features_upsampled, target_upsampled = shuffle(features_upsampled, target_upsampled, random_state=12345)
    
    return features_upsampled, target_upsampled

In [ ]:
# создадим переменные features_upsampled и target_upsampled
features_upsampled, target_upsampled = upsample(features_train, target_train, 4)  
# в данной ситуации repeat равно 4, так как именно в данное количество раз количество клиентов, которые остались,
# превышает число клиентов, которые ушли.

In [ ]:
model_1 = LogisticRegression(random_state=12345, solver='liblinear')
model_2 = DecisionTreeClassifier(**best_tree_model.get_params())
model_3 = RandomForestClassifier(**best_forest_model.get_params())
calc_f1(model_1, features_upsampled, target_upsampled)
calc_f1(model_2, features_upsampled, target_upsampled)
calc_f1(model_3, features_upsampled, target_upsampled)

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

**Уменьшение выборки**

In [ ]:
# создадим функцию upsample для уменьшения выборки
def downsample(features, target, fraction):
    features_zeros = features[target==0]
    features_ones = features[target==1]
    target_zeros = target[target==0]
    target_ones = target[target==1]
    
    features_downsampled = pd.concat([features_zeros.sample(frac=fraction, random_state=12345)] + [features_ones])
    target_downsampled = pd.concat([target_zeros.sample(frac=fraction, random_state=12345)] + [target_ones])
    
    features_downsampled, target_downsampled = shuffle(features_downsampled, target_downsampled, random_state=12345)
    
    return features_downsampled, target_downsampled

In [ ]:
# создадим переменные features_downsampled и target_downsampled
features_downsampled, target_downsampled = downsample(features_train, target_train, 0.25)  

In [ ]:
# применим к моделям функцию calc_f1 с учетом уменьшенной выборки
calc_f1(model_1, features_downsampled, target_downsampled)
calc_f1(model_2, features_downsampled, target_downsampled)
calc_f1(model_3, features_downsampled, target_downsampled)

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

Далее уточняем параметры случайного леса по валидации. Это заранее ограниченное семейство финальной модели; сравнение подходов выше имеет исследовательский характер.

In [ ]:
from sklearn.model_selection import ParameterGrid
results = []
for params in ParameterGrid({'max_depth': [3, 6, 9, 12], 'n_estimators': [100, 150, 200], 'class_weight': [None, 'balanced']}):
    candidate = RandomForestClassifier(random_state=12345, **params).fit(features_train, target_train)
    results.append((f1_score(target_valid, candidate.predict(features_valid)), candidate))
best_forest_model = max(results, key=lambda item: item[0])[1]
print(best_forest_model.get_params())

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

In [ ]:
# Параметры выбраны единым перебором выше, без повторного сужения диапазона.

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

In [ ]:
# Параметры выбраны единым перебором выше, без повторного сужения диапазона.

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

In [ ]:
# Параметры выбраны единым перебором выше, без повторного сужения диапазона.

In [ ]:
# Параметры выбраны единым перебором выше, без повторного сужения диапазона.

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

## Тестирование модели

Проверим нашу модель на тестовой выборке.

In [ ]:
# стандартизируем тестовые данные
features_test[numeric] = scaler.transform(features_test[numeric])

In [ ]:
model = best_forest_model
predictions_test = model.predict(features_test)
print('Test F1:', f1_score(target_test, predictions_test))
print('Test precision:', precision_score(target_test, predictions_test, zero_division=0))
print('Test recall:', recall_score(target_test, predictions_test, zero_division=0))

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

**Исследование метрики AUC-ROC**

In [ ]:
# построим ROC-кривую для случайного леса
# Используем уже выбранную модель.

probabilities_test = model.predict_proba(features_test)
probabilities_one_test = probabilities_test[:, 1]
fpr, tpr, thresholds = roc_curve(target_test, probabilities_one_test)

plt.figure()
plt.plot([0, 1], [0, 1], linestyle='--')
plt.plot(fpr, tpr)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-кривая')
plt.show()

In [ ]:
# посчитаем для случайного леса значение AUC-ROC
print('AUC-ROC:', roc_auc_score(target_test, probabilities_one_test))

Численные результаты и сравнения определяются повторным запуском, а не историческими значениями.

**Ограничения.** Прежние F1=0.64 и AUC=0.86 не подтверждены для исправленной версии. Единственная валидационная выборка используется многократно и может приводить к переобучению выбора параметров. Для новой оценки лучше фиксировать весь протокол заранее и повторить его на независимом временном срезе. Неизвестен момент сбора признаков относительно ухода клиента; причинные и временные утечки по этому набору без документации исключить нельзя.